# F-тест: структурные изменения/сдвиги

In [ ]:
import pandas as pd
import numpy as np

from statsmodels.formula.api import ols
from statsmodels.iolib.summary2 import summary_col # вывод результатов тестирования
from scipy.stats import f # F-распределение и критические значения

# Не показывать Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)

<div style="background-color:Bisque; color:DarkBlue; padding:30px;">

<i><b><span style="color: purple">Значимость структурных изменений/сдвигов</span></b></i><br>

Пусть бинарная переменная $D$ отвечает за структурное изменение (сдвиги).

Для учёта структурных изменений (сдвигов) рассмотрим две модели

__(1) Модель без структурных изменений (сдвигов)__

$$
(\log)y=\beta_0+\beta_1(\log)x_1+\cdots+\beta_k(\log)x_k+u=x^\top \beta+u
$$

Обозначим коэффициент детерминации $R^2$ для этой модели через $R^2_{res}$.

__(2) Модель со структурными изменениями (сдвигами)__

$$
(\log)y=x^\top \beta+(D\cdot x^\top) \gamma+v
$$

Обозначим коэффициент детерминации $R^2$ для этой модели через $R^2_{unres}$.

Для регрессии (2)  тестируем гипотезу, что $J=k+1$ новых коэффициентов совместно равны нулю в генеральной совокупности.

$$
H_0:\gamma_{0}=\cdots=\gamma_{k}=0
$$

__Интерпретация__: включённые в модель новые регрессоры совместно не влияют на зависимую переменную (в генеральной совокупности)

__Тестовая статистика__ для выбранной оценки ковариационной матрицы $\hat{V}$ формула для F-статистики (всё вычисляется автоматически!)

$$
F=\frac{R^2_{unres}-R^2_{res}}{1-R^2_{unres}}\cdot \frac{n-2(k+1)}{J}
$$

__Критическое значение__: $F$-распределения со степенями свободы ($k$ - число объясняющих переменных в модели (1))

$$
\begin{aligned}
df_1&=dfn=J=k+1 & df_2&=dfd=n-2(k+1)
\end{aligned}
$$

__Замечание__:
* для F-теста есть неробастный (по умолчанию) и робастный вариант
* для подогнанной __регрессии со структурными изменениями__ используем метод `.f_test(r_matrix)` или `.wald_test(r_matrix, use_f=True)`
* тестируемую гипотезу специфицируем в виде `'x1=x2=0'` или `'x1=0,x2=0'` (параметр `r_matrix`)

__Вывод__

**<span style="color:purple">Способ 1:</span>** используем $F$-статистику

* Отвергаем гипотезу $H_0$ при $F>F_{cr}$, <i><b><span style="color: purple">структурные изменения значимы</span></b></i><br>
* Не отвергаем гипотезу $H_0$ при $F<F_{cr}$, <i><b><span style="color: purple">структурные изменения незначимы</span></b></i><br>

**<span style="color:purple">Способ 2:</span>** используем P-значение

* Отвергаем гипотезу $H_0$ при $P<\alpha$, <i><b><span style="color: purple">структурные изменения значимы</span></b></i><br>
* Не отвергаем гипотезу $H_0$ при $P>\alpha$, <i><b><span style="color: purple">структурные изменения незначимы</span></b></i><br>

</div>

In [ ]:
# импорт данных
df = pd.read_csv('../datasets/sleep75.csv')

Для набора данных `sleep75` рассмотрим регрессию

$$
sleep=\beta_0+\beta_1 totwrk+\beta_2 age+\beta_3 smsa+error
$$

и регрессию со структурными сдвигами относительно гендерной бинарной переменной __male__

$$
sleep=\beta_0+\beta_1 totwrk+\beta_2 age+\beta_3 smsa+\beta_4 male+\beta_5 totwrk\cdot male+\beta_6 age\cdot male+\beta_7 smsa\cdot male+error
$$

Тестируем значимость структурных сдвигов (т.е. тестируем совместную значимость __male, totwrk:male, age:male, smsa:male__ для регрессии со структурными изменениями)

$$
H_0:\beta_{male}=\beta_{totwrk:male}=\beta_{age:male}=\beta_{smsa:male}=0
$$

Для этой гипотезы $J=4$

## F-тест (неробастный)

Используем OLS-оценку ковариационной матрицы $\hat{V}=s^2(X^\top X)^{-1}$

In [ ]:
# спецификация исходной модели
mod = ols(formula='sleep~1+totwrk+age+smsa', data=df)
# подгонка исходной модели с неробастной оценкой ковариационной матрицы
res_ols = mod.fit()
# спецификация модели со структурными сдвигами
mod_breaks = ols(formula='sleep~1+totwrk+age+smsa+male+totwrk:male+age:male+smsa:male', data=df)
# подгонка модели со структурными сдвигами с неробастной оценкой ковариационной матрицы
res_breaks_ols = mod_breaks.fit()
# результаты двух оцениваний в одной таблице
summary_col([res_ols, res_breaks_ols], stars=True,
            regressor_order=['totwrk', 'age', 'smsa', 'male', 'totwrk:male', 'age:male', 'smsa:male'])

In [ ]:
# спецификация тестируемой гипотезы
F_test = res_breaks_ols.f_test('male=0, totwrk:male=0, age:male=0, smsa:male=0')
print(F_test)

In [ ]:
# тестовая статистика и её P-значение
F_test.statistic, F_test.pvalue

In [ ]:
# степени свободы для F-распределения
F_test.df_num, F_test.df_denom

In [ ]:
# уровень значимости
sign_level = 0.05
# критическое значение F-распределения
f.isf(q=sign_level, dfn=F_test.df_num, dfd=F_test.df_denom)

In [ ]:
# альтернативно
W_test = res_breaks_ols.wald_test('male=0, totwrk:male=0, age:male=0, smsa:male=0', use_f=True)
print(W_test)

**ВЫВОД**: на уровне значимости 5% гипотеза $H_0$ не отвергается (т.к. $F<F_{cr}$ и $P>\alpha$), структурные изменения незначимы

## F-тест (робастный)

Используем HC3-оценку ковариационной матрицы

In [ ]:
# спецификация исходной модели
mod = ols(formula='sleep~1+totwrk+age+smsa', data=df)
# подгонка исходной модели с робастной HC3-оценкой ковариационной матрицы
res_hc = mod.fit(cov_type='HC3', use_t=True)
# спецификация модели со структурными сдвигами
mod_breaks = ols(formula='sleep~1+totwrk+age+smsa+male+totwrk:male+age:male+smsa:male', data=df)
# подгонка модели со структурными сдвигами с робастной HC3-оценкой ковариационной матрицы
res_breaks_hc = mod_breaks.fit(cov_type='HC3', use_t=True)
# результаты двух оцениваний в одной таблице
summary_col([res_hc, res_breaks_hc], stars=True,
            regressor_order=['totwrk', 'age', 'smsa', 'male', 'totwrk:male', 'age:male', 'smsa:male'])

In [ ]:
# спецификация тестируемой гипотезы
F_test = res_breaks_hc.f_test('male=0, totwrk:male=0, age:male=0, smsa:male=0')
print(F_test)

In [ ]:
# тестовая статистика и её P-значение
F_test.statistic, F_test.pvalue

In [ ]:
# уровень значимости
sign_level = 0.05
# критическое значение F-распределения
f.isf(q=sign_level, dfn=F_test.df_num, dfd=F_test.df_denom)

In [ ]:
# альтернативно
W_test = res_breaks_hc.wald_test('male=0, totwrk:male=0, age:male=0, smsa:male=0', use_f=True)
print(W_test)

**ВЫВОД**: на уровне значимости 5% гипотеза $H_0$ не отвергается (т.к. $F<F_{cr}$ и $P>\alpha$), структурные изменения незначимы